# Random forest — frozen evidence

Model regeneration under scikit-learn 1.8.0 is documented in Git history; no new test optimization occurs here.

This notebook reads frozen artifacts only. See README for separate retraining commands.

In [1]:
from pathlib import Path
import json
from IPython.display import display
from integration.audit import read_csv

ROOT = Path.cwd()
assert (ROOT / "data/splits/test.csv").exists(), "Run from repository root"
OUT = ROOT / "results" / 'rf'
protocol = json.loads((OUT / "protocol_frozen.json").read_text())
print(protocol.get("Evidence_status", "Historical frozen experiment; see integration audit for chronology"))
display(protocol["Selected_parameters"])

Historical frozen experiment; see integration audit for chronology


{'n_estimators': 500,
 'min_samples_split': 5,
 'min_samples_leaf': 2,
 'max_features': 0.5,
 'max_depth': 8,
 'class_weight': {'0': 1, '1': 3}}

In [2]:
display(read_csv(OUT / "cv_results.csv").sort_values("Mean_CV_AP", ascending=False).head())
display(read_csv(OUT / "test_metrics.csv")[["Model","Threshold","AP","ROC-AUC","Precision","Recall","F1","Accuracy","TN","FP","FN","TP","Cost_5","Alert_rate"]])

,Candidate,Parameters,Mean_CV_AP,SD_CV_AP,Mean_fit_AP,Mean_CV_ROC_AUC,Mean_fit_seconds,Fold_0_AP,Fold_1_AP,Fold_2_AP,Fold_3_AP,Fold_4_AP
3,3,"{""class_weight"": {""0"": 1, ""1"": 3}, ""max_depth""...",0.564717,0.013627,0.682997,0.779972,22.222557,0.573051,0.568018,0.572111,0.569808,0.540597
13,13,"{""class_weight"": {""0"": 1, ""1"": 3}, ""max_depth""...",0.564713,0.013252,0.682645,0.780071,13.538270,0.573677,0.568697,0.573036,0.566542,0.541613
10,10,"{""class_weight"": {""0"": 1, ""1"": 3}, ""max_depth""...",0.564501,0.013480,0.679785,0.779550,13.349472,0.576242,0.566289,0.569268,0.569419,0.541287
1,1,"{""class_weight"": null, ""max_depth"": 8, ""max_fe...",0.563746,0.015481,0.689783,0.780045,6.894008,0.573862,0.562738,0.574088,0.570741,0.537299
6,6,"{""class_weight"": null, ""max_depth"": 12, ""max_f...",0.560732,0.015634,0.859766,0.779700,31.675847,0.572016,0.561294,0.572625,0.563471,0.534254


,Model,Threshold,AP,ROC-AUC,Precision,Recall,F1,Accuracy,TN,FP,FN,TP,Cost_5,Alert_rate
0,Baseline RF,0.500000,0.531169,0.753432,0.644133,0.380558,0.478446,0.816500,4394,279,822,505,4389,0.130667
1,Tuned RF,0.500000,0.550755,0.775246,0.546318,0.519970,0.532819,0.798333,4100,573,637,690,3758,0.210500
2,Tuned RF / cost ratio 1,0.706812,0.550755,0.775246,0.658501,0.344386,0.452251,0.815500,4436,237,870,457,4587,0.115667
3,Tuned RF / cost ratio 3,0.452583,0.550755,0.775246,0.502913,0.585531,0.541086,0.780333,3905,768,550,777,3518,0.257500
4,Tuned RF / cost ratio 5,0.322881,0.550755,0.775246,0.351909,0.770912,0.483231,0.635333,2789,1884,304,1023,3404,0.484500
5,Tuned RF / cost ratio 10,0.248974,0.550755,0.775246,0.288916,0.903542,0.437831,0.486833,1722,2951,128,1199,3591,0.691667
6,Always negative,1.000000,0.221167,0.500000,0.000000,0.000000,0.000000,0.778833,4673,0,1327,0,6635,0.000000
7,Always positive,0.000000,0.221167,0.500000,0.221167,1.000000,0.362222,0.221167,0,4673,0,1327,4673,1.000000


In [3]:
display(read_csv(OUT / "bootstrap_intervals.csv"))
display(read_csv(OUT / "validation_importance.csv").head(10))

,Quantity,Estimate,CI_low,CI_high,Bootstrap_replicates
0,Tuned AP,0.550755,0.525705,0.575176,1000
1,Tuned ROC-AUC,0.775246,0.760381,0.790274,1000
2,AP difference: tuned - baseline,0.019586,0.007525,0.032737,1000
3,ROC-AUC difference: tuned - baseline,0.021813,0.013581,0.030554,1000
4,Cost/1000 difference: tuned r=5 threshold - tu...,-59.000000,-85.500000,-31.500000,1000


,Feature,Mean_AP_decrease,SD_AP_decrease
0,PAY_0,0.197072,0.005338
1,PAY_2,0.019572,0.002568
2,PAY_3,0.010870,0.001954
3,LIMIT_BAL,0.010002,0.001749
4,BILL_AMT1,0.008749,0.002516
5,PAY_4,0.007584,0.002347
6,PAY_6,0.006380,0.001705
7,PAY_AMT3,0.005807,0.001756
8,PAY_AMT2,0.005023,0.001507
9,PAY_5,0.004610,0.001309


AP is average_precision_score, not trapezoidal PR-AUC. Bootstrap intervals condition on fixed predictions, model and thresholds; no retraining or threshold-selection uncertainty is included. Importance is associative, not causal.